# cjm-media-plugin-lavasr

> LavaSR speech enhancement plugin for the cjm-plugin-system that provides bandwidth extension and denoising to improve speech audio quality before transcription.

## Install

```bash
pip install cjm_media_plugin_lavasr
```

## Project Structure

```
nbs/
├── meta.ipynb   # Metadata introspection for the LavaSR plugin used by cjm-ctl to generate the registration manifest.
└── plugin.ipynb # LavaSR v2 speech enhancement plugin — provides bandwidth extension and optional denoising to improve speech audio quality before transcription.
```

Total: 2 notebooks

## Module Dependencies

```mermaid
graph LR
    meta["meta<br/>Metadata"]
    plugin["plugin<br/>Plugin"]

```

No cross-module dependencies detected.

## CLI Reference

No CLI commands found in this project.

## Module Overview

Detailed documentation for each module in the project:

### Metadata (`meta.ipynb`)
> Metadata introspection for the LavaSR plugin used by cjm-ctl to generate the registration manifest.

#### Import

```python
from cjm_media_plugin_lavasr.meta import (
    get_plugin_metadata
)
```

#### Functions

```python
def get_plugin_metadata() -> Dict[str, Any]:  # Plugin metadata for manifest generation
    """Return metadata required to register this plugin with the PluginManager."""
    # Fallback base path (current behavior for backward compatibility)
    base_path = os.path.dirname(os.path.dirname(sys.executable))
    
    # Use CJM config if available, else fallback to env-relative paths
    cjm_data_dir = os.environ.get("CJM_DATA_DIR")
    
    # Plugin data directory
    plugin_name = "cjm-media-plugin-lavasr"
    if cjm_data_dir
    "Return metadata required to register this plugin with the PluginManager."
```


### Plugin (`plugin.ipynb`)
> LavaSR v2 speech enhancement plugin — provides bandwidth extension and optional denoising to improve speech audio quality before transcription.

#### Import

```python
from cjm_media_plugin_lavasr.plugin import (
    LavaSRPluginConfig,
    LavaSRProcessingPlugin
)
```

#### Functions

```python
@patch
def _apply_config(self:LavaSRProcessingPlugin,
                  config: Optional[Any] = None,  # Configuration dict or None for defaults
                 ) -> None
    """
    CR-4: apply config values only. Called by initialize (first-time) and the
    substrate's reconfigure delta path. Model release on a model_path/device change
    is handled declaratively via RELOAD_TRIGGER -> _release_model (device resolved
    lazily in _load_model).
    """
```

```python
@patch
def prefetch(self:LavaSRProcessingPlugin) -> None
    """
    CR-4 (SG-19): eagerly load the model so the first execute() doesn't pay
    the download/load cost. Idempotent via _load_model's None-guard.
    """
```

```python
@patch
def on_disable(self:LavaSRProcessingPlugin) -> None
    """
    CR-2: release the GPU model when the operator disables the plugin (the
    worker stays alive); lazy reload on the next execute after re-enable.
    """
```

```python
@patch
def cleanup(self:LavaSRProcessingPlugin) -> None
    "Clean up plugin resources."
```

```python
@patch
def is_available(self:LavaSRProcessingPlugin) -> bool:  # Whether the plugin can run
    """Check if the plugin is available on this system."""
    try
    "Check if the plugin is available on this system."
```

```python
@patch
def _load_model(self:LavaSRProcessingPlugin) -> None:
    """Load the LavaSR v2 model (lazy, cached).

    CR-4: a model_path/device change releases the model declaratively via
    RELOAD_TRIGGER -> _release_model, so a None model means a (re)load is required.

    The heartbeat wraps the WHOLE load. LavaEnhance2's constructor downloads weights
    from HF Hub on a cold cache and that download is silent to the substrate's stall
    detector; worse, the constructor only triggers it for the "YatharthS/LavaSR"
    sentinel via an un-monitored snapshot_download. So we pre-resolve the snapshot via
    snapshot_download_with_progress (monitored) and hand the LOCAL path to
    LavaEnhance2 — bypassing its internal download. An operator-supplied local path
    (not the sentinel) passes straight through."""
    if self._model is not None
    """
    Load the LavaSR v2 model (lazy, cached).
    
    CR-4: a model_path/device change releases the model declaratively via
    RELOAD_TRIGGER -> _release_model, so a None model means a (re)load is required.
    
    The heartbeat wraps the WHOLE load. LavaEnhance2's constructor downloads weights
    from HF Hub on a cold cache and that download is silent to the substrate's stall
    detector; worse, the constructor only triggers it for the "YatharthS/LavaSR"
    sentinel via an un-monitored snapshot_download. So we pre-resolve the snapshot via
    snapshot_download_with_progress (monitored) and hand the LOCAL path to
    LavaEnhance2 — bypassing its internal download. An operator-supplied local path
    (not the sentinel) passes straight through.
    """
```

```python
@patch
def _release_model(self:LavaSRProcessingPlugin) -> None
    """
    CR-4: release the LavaSR model + free CUDA cache (cjm-torch-plugin-utils).
    RELOAD_TRIGGER target for model_path/device; on_disable / cleanup delegate here.
    Idempotent via release_model (no-op when already released).
    """
```

```python
@patch
def _store_job(self:LavaSRProcessingPlugin,
    """
    Hash input/output files and store a processing job record (upsert by
    action + input_path + config_hash; logs + swallows save failures).
    """
```

```python
@patch
def _action_get_info(self:LavaSRProcessingPlugin, **kwargs) -> Dict[str, Any]
    "Action wrapper -> get_info()."
```

```python
@patch
def _action_enhance_speech(self:LavaSRProcessingPlugin, **kwargs) -> Dict[str, Any]
    "Action wrapper -> _enhance_speech()."
```

```python
@patch
def _enhance_speech(self:LavaSRProcessingPlugin,
    "Enhance speech quality via bandwidth extension and optional denoising."
```

#### Classes

```python
@dataclass
class LavaSRPluginConfig(HFCacheConfig):
    """
    Configuration for the LavaSR speech enhancement plugin.
    
    Composes HFCacheConfig (cache_dir / revision / local_files_only / trust_remote_code,
    each RELOAD_TRIGGER-tagged) so the HF Hub snapshot download used to resolve the
    LavaSR weights is operator-controllable.
    """
    
    model_path: str = field(...)
    device: str = field(...)
    denoise: bool = field(...)
    enhance: bool = field(...)
    batch_mode: bool = field(...)
    input_sr: int = field(...)
    cutoff: Optional[int] = field(...)
    output_format: str = field(...)
```

```python
class LavaSRProcessingPlugin:
    def __init__(self):
        """Initialize the plugin."""
        self.logger = logging.getLogger(f"{__name__}.{type(self).__name__}")
        self.config: Optional[LavaSRPluginConfig] = None
    "LavaSR v2 speech enhancement plugin for bandwidth extension and denoising."
    
    def __init__(self):
            """Initialize the plugin."""
            self.logger = logging.getLogger(f"{__name__}.{type(self).__name__}")
            self.config: Optional[LavaSRPluginConfig] = None
        "Initialize the plugin."
    
    def name(self) -> str:  # Plugin name identifier
            """Get the plugin name."""
            return "cjm-media-plugin-lavasr"
        
        @property
        def version(self) -> str:  # Plugin version string
        "Get the plugin name."
    
    def version(self) -> str:  # Plugin version string
            """Get the plugin version."""
            from cjm_media_plugin_lavasr import __version__
            return __version__
        
        @property
        def supported_media_types(self) -> List[str]:  # Supported media types
        "Get the plugin version."
    
    def supported_media_types(self) -> List[str]:  # Supported media types
            """Get supported media types."""
            return ["audio"]
        
        # ── Lifecycle ────────────────────────────────────────────────────
        
    
        def initialize(self,
                       config: Optional[Any] = None,  # Configuration dict or None for defaults
                      ) -> None
        "Get supported media types."
    
    def initialize(self,
                       config: Optional[Any] = None,  # Configuration dict or None for defaults
                      ) -> None
        "First-time setup. CR-4: config application factored into _apply_config; the
substrate's reconfigure path fires _release_model on a model_path/device change
then re-applies config."
    
    def get_config_schema(self) -> Dict[str, Any]:  # JSON Schema for UI forms
            """Return JSON Schema for the plugin configuration."""
            return dataclass_to_jsonschema(LavaSRPluginConfig)
        
        def get_current_config(self) -> Dict[str, Any]:  # Current config as dict
        "Return JSON Schema for the plugin configuration."
    
    def get_current_config(self) -> Dict[str, Any]:  # Current config as dict
            """Return the current configuration."""
            return config_to_dict(self.config) if self.config else {}
        
        # ── Model Management ────────────────────────────────────────────
        
        
        
        # ── Job Storage ──────────────────────────────────────────────────
        
        
        # ── Action Dispatch ──────────────────────────────────────────────
        
        def execute(self,
                    action: str = "enhance_speech",  # Action to perform
                    **kwargs
                   ) -> Dict[str, Any]:  # Action result
        "Return the current configuration."
    
    def execute(self,
                    action: str = "enhance_speech",  # Action to perform
                    **kwargs
                   ) -> Dict[str, Any]:  # Action result
        "Dispatch to the `@plugin_action`-tagged handler for `action` (SG-44)."
    
    def get_info(self,
                     file_path: str,  # Path to audio file
                    ) -> MediaMetadata:  # File metadata
        "Get basic audio file metadata via soundfile."
```
